# Conversational AI 
## Assignment 2 – Hybrid RAG System with Automated Evaluation
### Group 122
### Group Member Details:
|S.No|   Group Member Names        |      BITS Email ID                        | Contribution in %  |
|----|-----------------------------|-------------------------------------------|--------------------|
|1   | **SK SHAHRUKH SABA**        |     2024aa05401@wilp.bits-pilani.ac.in    | 100%               |
|2   | **SANKHA CHAKRABORTY**      |     2024AA05393@wilp.bits-pilani.ac.in    | 100%               |
|3   | **NEELASHA ROY**            |     2024aa05698@wilp.bits-pilani.ac.in    | 100%               |
|4   | **ARUNAVA DUTTA**           |     2024aa05374@wilp.bits-pilani.ac.in    | 100%               |
|5   | **CHINTAN DESAI**           |     2024aa05648@wilp.bits-pilani.ac.in    | 100%               |

# Step 1.0 Wikipedia Dataset Creator

This notebook creates two JSON datasets from Wikipedia pages:
1. **fixed_url.json** - Contains data from 200 predefined Wikipedia URLs across 8 domains
2. **random_url.json** - Contains data from 300 randomly selected Wikipedia URLs

Each dataset entry includes:
- `id`: A unique GUID identifier
- `url`: The Wikipedia page URL
- `domain`: Category/domain of the page (e.g., people, places, events, technology, science, arts, sports, organization)
- `content`: Up to 200 words of raw text from the page

## Import Required Libraries

This cell imports all the necessary libraries for:
- `requests`: Making HTTP requests to fetch Wikipedia pages
- `BeautifulSoup`: Parsing HTML content from web pages
- `json`: Reading and writing JSON files
- `uuid`: Generating unique identifiers (GUIDs)
- `re`: Regular expressions for text cleaning
- `time`: Adding delays between requests to respect server rate limits

In [1]:
import requests
from bs4 import BeautifulSoup
import json
import uuid
import re
import time

## Helper Function: Extract Text from Wikipedia URL

This function takes a Wikipedia URL and extracts the main content text from the page.
It performs the following operations:
1. Fetches the HTML content from the URL
2. Parses the HTML using BeautifulSoup
3. Extracts text from paragraph tags within the main content area
4. Cleans the text by removing extra whitespace and reference markers
5. Limits the output to the first 200 words

In [2]:
def extract_text_from_wikipedia(url, max_words=200):
    """
    Extracts up to max_words of text content from a Wikipedia page.
    
    Args:
        url (str): The Wikipedia page URL
        max_words (int): Maximum number of words to extract (default: 200)
    
    Returns:
        str: Extracted text content limited to max_words
    """
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Find the main content div
        content_div = soup.find('div', {'id': 'mw-content-text'})
        
        if not content_div:
            return "Content not found"
        
        # Extract text from paragraphs
        paragraphs = content_div.find_all('p')
        text_parts = []
        
        for p in paragraphs:
            text = p.get_text()
            # Remove reference markers like [1], [2], etc.
            text = re.sub(r'\[\d+\]', '', text)
            # Remove extra whitespace
            text = ' '.join(text.split())
            if text:
                text_parts.append(text)
        
        full_text = ' '.join(text_parts)
        
        # Limit to max_words
        words = full_text.split()
        limited_text = ' '.join(words[:max_words])
        
        return limited_text
    
    except Exception as e:
        return f"Error extracting content: {str(e)}"

## Helper Function: Generate Unique ID

This function generates a unique identifier (GUID/UUID) for each dataset entry.
It uses Python's uuid4() function which creates a random UUID.

In [3]:
def generate_unique_id():
    """
    Generates a unique GUID/UUID identifier.
    
    Returns:
        str: A unique identifier string
    """
    return str(uuid.uuid4())

## Helper Function: Create Dataset Entry

This function creates a single dataset entry dictionary with all required fields:
- Generates a unique ID
- Stores the URL
- Assigns the domain/category
- Extracts and stores the content (up to 200 words)

In [4]:
def create_dataset_entry(url, domain):
    """
    Creates a dataset entry dictionary for a Wikipedia page.
    
    Args:
        url (str): The Wikipedia page URL
        domain (str): The category/domain of the page
    
    Returns:
        dict: A dictionary containing id, url, domain, and content
    """
    print(f"Processing: {url}")
    
    entry = {
        "id": generate_unique_id(),
        "url": url,
        "domain": domain,
        "content": extract_text_from_wikipedia(url)
    }
    
    # Add a small delay to be respectful to Wikipedia servers
    time.sleep(1)
    
    return entry

## Helper Function: Save Dataset to JSON

This function saves a list of dataset entries to a JSON file.
It formats the JSON with proper indentation for readability.

In [5]:
def save_to_json(data, filename):
    """
    Saves a list of dataset entries to a JSON file.
    
    Args:
        data (list): List of dataset entry dictionaries
        filename (str): Name of the output JSON file
    """
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    
    print(f"\nDataset saved to {filename}")
    print(f"Total entries: {len(data)}")

## Helper Function: Get Random Wikipedia URL

This function fetches a random Wikipedia article URL using Wikipedia's random article feature.
It follows the redirect from the special random page to get the actual article URL.

In [6]:
def get_random_wikipedia_url():
    """
    Fetches a random Wikipedia article URL.
    
    Returns:
        str: URL of a random Wikipedia article
    """
    random_url = "https://en.wikipedia.org/wiki/Special:Random"
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    
    try:
        response = requests.get(random_url, headers=headers, timeout=10)
        return response.url
    except Exception as e:
        print(f"Error getting random URL: {e}")
        return None

## Helper Function: Detect Domain Category

This function analyzes a Wikipedia page to determine its domain/category.
It examines the page's categories and content to classify it into one of several domains:
- people: Biographical articles
- places: Geographic locations
- events: Historical events
- science: Scientific topics
- technology: Technology-related articles
- arts: Art, music, literature
- sports: Sports-related content
- organization: Companies, institutions
- other: Everything else

In [7]:
def detect_domain(url):
    """
    Detects the domain/category of a Wikipedia page.
    
    Args:
        url (str): The Wikipedia page URL
    
    Returns:
        str: The detected domain category
    """
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Get categories from the page
        categories_div = soup.find('div', {'id': 'mw-normal-catlinks'})
        categories_text = categories_div.get_text().lower() if categories_div else ""
        
        # Get page title and content for additional context
        title = soup.find('h1', {'id': 'firstHeading'})
        title_text = title.get_text().lower() if title else ""
        
        # Domain detection based on keywords
        people_keywords = ['birth', 'death', 'born', 'people', 'politician', 'actor', 'actress', 
                          'singer', 'writer', 'artist', 'scientist', 'player', 'biography']
        places_keywords = ['city', 'country', 'town', 'village', 'geography', 'location', 
                          'region', 'state', 'province', 'district', 'mountain', 'river']
        events_keywords = ['war', 'battle', 'revolution', 'event', 'history', 'incident', 
                          'disaster', 'election', 'championship', 'tournament']
        science_keywords = ['science', 'biology', 'chemistry', 'physics', 'mathematics', 
                           'medicine', 'astronomy', 'geology']
        technology_keywords = ['technology', 'software', 'computer', 'internet', 'programming',
                              'engineering', 'device', 'invention']
        arts_keywords = ['art', 'music', 'film', 'movie', 'album', 'song', 'book', 'novel',
                        'painting', 'literature', 'theatre', 'dance']
        sports_keywords = ['sport', 'football', 'basketball', 'cricket', 'tennis', 'olympics',
                          'athlete', 'team', 'league', 'match']
        org_keywords = ['company', 'organization', 'corporation', 'university', 'institution',
                       'foundation', 'agency', 'association']
        
        combined_text = categories_text + " " + title_text
        
        # Check each domain
        if any(kw in combined_text for kw in people_keywords):
            return "people"
        elif any(kw in combined_text for kw in places_keywords):
            return "places"
        elif any(kw in combined_text for kw in events_keywords):
            return "events"
        elif any(kw in combined_text for kw in science_keywords):
            return "science"
        elif any(kw in combined_text for kw in technology_keywords):
            return "technology"
        elif any(kw in combined_text for kw in arts_keywords):
            return "arts"
        elif any(kw in combined_text for kw in sports_keywords):
            return "sports"
        elif any(kw in combined_text for kw in org_keywords):
            return "organization"
        else:
            return "other"
            
    except Exception as e:
        return "other"

## Define Fixed Wikipedia URLs

This cell defines 200 predefined Wikipedia URLs covering various domains:
- **People** (50 URLs): Scientists, artists, historical figures, athletes, leaders
- **Places** (30 URLs): Cities, countries, natural landmarks, geographic features
- **Events** (25 URLs): Historical events, wars, revolutions, discoveries
- **Technology** (25 URLs): Computing, internet, inventions, engineering
- **Science** (25 URLs): Physics, biology, chemistry, astronomy, medicine
- **Arts** (20 URLs): Paintings, music, literature, films, architecture
- **Sports** (15 URLs): Sports, competitions, athletes, Olympic events
- **Organization** (10 URLs): Companies, institutions, organizations

Each URL is paired with its domain category.

In [8]:
# Fixed Wikipedia URLs with their domains (200 URLs)
fixed_urls = [
    # ===== PEOPLE (50 URLs) =====
    # Scientists
    ("https://en.wikipedia.org/wiki/Albert_Einstein", "people"),
    ("https://en.wikipedia.org/wiki/Marie_Curie", "people"),
    ("https://en.wikipedia.org/wiki/Isaac_Newton", "people"),
    ("https://en.wikipedia.org/wiki/Charles_Darwin", "people"),
    ("https://en.wikipedia.org/wiki/Nikola_Tesla", "people"),
    ("https://en.wikipedia.org/wiki/Stephen_Hawking", "people"),
    ("https://en.wikipedia.org/wiki/Galileo_Galilei", "people"),
    ("https://en.wikipedia.org/wiki/Richard_Feynman", "people"),
    ("https://en.wikipedia.org/wiki/Ada_Lovelace", "people"),
    ("https://en.wikipedia.org/wiki/Alan_Turing", "people"),
    # Historical Leaders
    ("https://en.wikipedia.org/wiki/Abraham_Lincoln", "people"),
    ("https://en.wikipedia.org/wiki/Winston_Churchill", "people"),
    ("https://en.wikipedia.org/wiki/Mahatma_Gandhi", "people"),
    ("https://en.wikipedia.org/wiki/Nelson_Mandela", "people"),
    ("https://en.wikipedia.org/wiki/Martin_Luther_King_Jr.", "people"),
    ("https://en.wikipedia.org/wiki/Napoleon", "people"),
    ("https://en.wikipedia.org/wiki/Cleopatra", "people"),
    ("https://en.wikipedia.org/wiki/Julius_Caesar", "people"),
    ("https://en.wikipedia.org/wiki/Queen_Victoria", "people"),
    ("https://en.wikipedia.org/wiki/Alexander_the_Great", "people"),
    # Artists & Writers
    ("https://en.wikipedia.org/wiki/Leonardo_da_Vinci", "people"),
    ("https://en.wikipedia.org/wiki/Vincent_van_Gogh", "people"),
    ("https://en.wikipedia.org/wiki/Pablo_Picasso", "people"),
    ("https://en.wikipedia.org/wiki/Michelangelo", "people"),
    ("https://en.wikipedia.org/wiki/William_Shakespeare", "people"),
    ("https://en.wikipedia.org/wiki/Wolfgang_Amadeus_Mozart", "people"),
    ("https://en.wikipedia.org/wiki/Ludwig_van_Beethoven", "people"),
    ("https://en.wikipedia.org/wiki/Johann_Sebastian_Bach", "people"),
    ("https://en.wikipedia.org/wiki/Frida_Kahlo", "people"),
    ("https://en.wikipedia.org/wiki/Claude_Monet", "people"),
    # Modern Figures
    ("https://en.wikipedia.org/wiki/Elon_Musk", "people"),
    ("https://en.wikipedia.org/wiki/Steve_Jobs", "people"),
    ("https://en.wikipedia.org/wiki/Bill_Gates", "people"),
    ("https://en.wikipedia.org/wiki/Jeff_Bezos", "people"),
    ("https://en.wikipedia.org/wiki/Mark_Zuckerberg", "people"),
    ("https://en.wikipedia.org/wiki/Oprah_Winfrey", "people"),
    ("https://en.wikipedia.org/wiki/Barack_Obama", "people"),
    ("https://en.wikipedia.org/wiki/Angela_Merkel", "people"),
    ("https://en.wikipedia.org/wiki/Malala_Yousafzai", "people"),
    ("https://en.wikipedia.org/wiki/Greta_Thunberg", "people"),
    # Athletes
    ("https://en.wikipedia.org/wiki/Michael_Jordan", "people"),
    ("https://en.wikipedia.org/wiki/Lionel_Messi", "people"),
    ("https://en.wikipedia.org/wiki/Cristiano_Ronaldo", "people"),
    ("https://en.wikipedia.org/wiki/Serena_Williams", "people"),
    ("https://en.wikipedia.org/wiki/Usain_Bolt", "people"),
    ("https://en.wikipedia.org/wiki/Muhammad_Ali", "people"),
    ("https://en.wikipedia.org/wiki/Roger_Federer", "people"),
    ("https://en.wikipedia.org/wiki/LeBron_James", "people"),
    ("https://en.wikipedia.org/wiki/Tiger_Woods", "people"),
    ("https://en.wikipedia.org/wiki/Michael_Phelps", "people"),
    
    # ===== PLACES (30 URLs) =====
    # Natural Landmarks
    ("https://en.wikipedia.org/wiki/Grand_Canyon", "places"),
    ("https://en.wikipedia.org/wiki/Mount_Everest", "places"),
    ("https://en.wikipedia.org/wiki/Niagara_Falls", "places"),
    ("https://en.wikipedia.org/wiki/Great_Barrier_Reef", "places"),
    ("https://en.wikipedia.org/wiki/Amazon_rainforest", "places"),
    ("https://en.wikipedia.org/wiki/Sahara", "places"),
    ("https://en.wikipedia.org/wiki/Victoria_Falls", "places"),
    ("https://en.wikipedia.org/wiki/Yellowstone_National_Park", "places"),
    # Cities
    ("https://en.wikipedia.org/wiki/Tokyo", "places"),
    ("https://en.wikipedia.org/wiki/New_York_City", "places"),
    ("https://en.wikipedia.org/wiki/London", "places"),
    ("https://en.wikipedia.org/wiki/Paris", "places"),
    ("https://en.wikipedia.org/wiki/Rome", "places"),
    ("https://en.wikipedia.org/wiki/Sydney", "places"),
    ("https://en.wikipedia.org/wiki/Dubai", "places"),
    ("https://en.wikipedia.org/wiki/Singapore", "places"),
    ("https://en.wikipedia.org/wiki/Hong_Kong", "places"),
    ("https://en.wikipedia.org/wiki/Berlin", "places"),
    # Countries
    ("https://en.wikipedia.org/wiki/United_States", "places"),
    ("https://en.wikipedia.org/wiki/China", "places"),
    ("https://en.wikipedia.org/wiki/India", "places"),
    ("https://en.wikipedia.org/wiki/Japan", "places"),
    ("https://en.wikipedia.org/wiki/Brazil", "places"),
    ("https://en.wikipedia.org/wiki/Australia", "places"),
    ("https://en.wikipedia.org/wiki/Canada", "places"),
    ("https://en.wikipedia.org/wiki/Germany", "places"),
    ("https://en.wikipedia.org/wiki/France", "places"),
    ("https://en.wikipedia.org/wiki/United_Kingdom", "places"),
    ("https://en.wikipedia.org/wiki/Italy", "places"),
    ("https://en.wikipedia.org/wiki/Russia", "places"),
    
    # ===== EVENTS (25 URLs) =====
    ("https://en.wikipedia.org/wiki/World_War_II", "events"),
    ("https://en.wikipedia.org/wiki/World_War_I", "events"),
    ("https://en.wikipedia.org/wiki/French_Revolution", "events"),
    ("https://en.wikipedia.org/wiki/American_Revolution", "events"),
    ("https://en.wikipedia.org/wiki/Industrial_Revolution", "events"),
    ("https://en.wikipedia.org/wiki/Renaissance", "events"),
    ("https://en.wikipedia.org/wiki/Cold_War", "events"),
    ("https://en.wikipedia.org/wiki/Moon_landing", "events"),
    ("https://en.wikipedia.org/wiki/Fall_of_the_Berlin_Wall", "events"),
    ("https://en.wikipedia.org/wiki/September_11_attacks", "events"),
    ("https://en.wikipedia.org/wiki/Black_Death", "events"),
    ("https://en.wikipedia.org/wiki/Reformation", "events"),
    ("https://en.wikipedia.org/wiki/Russian_Revolution", "events"),
    ("https://en.wikipedia.org/wiki/American_Civil_War", "events"),
    ("https://en.wikipedia.org/wiki/Hiroshima_and_Nagasaki_atomic_bombings", "events"),
    ("https://en.wikipedia.org/wiki/COVID-19_pandemic", "events"),
    ("https://en.wikipedia.org/wiki/Great_Depression", "events"),
    ("https://en.wikipedia.org/wiki/Chernobyl_disaster", "events"),
    ("https://en.wikipedia.org/wiki/Titanic", "events"),
    ("https://en.wikipedia.org/wiki/Discovery_of_the_Americas", "events"),
    ("https://en.wikipedia.org/wiki/Ancient_Egypt", "events"),
    ("https://en.wikipedia.org/wiki/Roman_Empire", "events"),
    ("https://en.wikipedia.org/wiki/Ancient_Greece", "events"),
    ("https://en.wikipedia.org/wiki/Byzantine_Empire", "events"),
    ("https://en.wikipedia.org/wiki/Ottoman_Empire", "events"),
    
    # ===== TECHNOLOGY (25 URLs) =====
    ("https://en.wikipedia.org/wiki/Artificial_intelligence", "technology"),
    ("https://en.wikipedia.org/wiki/Internet", "technology"),
    ("https://en.wikipedia.org/wiki/Computer", "technology"),
    ("https://en.wikipedia.org/wiki/Smartphone", "technology"),
    ("https://en.wikipedia.org/wiki/World_Wide_Web", "technology"),
    ("https://en.wikipedia.org/wiki/Machine_learning", "technology"),
    ("https://en.wikipedia.org/wiki/Blockchain", "technology"),
    ("https://en.wikipedia.org/wiki/Cryptocurrency", "technology"),
    ("https://en.wikipedia.org/wiki/Bitcoin", "technology"),
    ("https://en.wikipedia.org/wiki/Electric_vehicle", "technology"),
    ("https://en.wikipedia.org/wiki/Robotics", "technology"),
    ("https://en.wikipedia.org/wiki/3D_printing", "technology"),
    ("https://en.wikipedia.org/wiki/Virtual_reality", "technology"),
    ("https://en.wikipedia.org/wiki/Augmented_reality", "technology"),
    ("https://en.wikipedia.org/wiki/Cloud_computing", "technology"),
    ("https://en.wikipedia.org/wiki/5G", "technology"),
    ("https://en.wikipedia.org/wiki/Quantum_computing", "technology"),
    ("https://en.wikipedia.org/wiki/Self-driving_car", "technology"),
    ("https://en.wikipedia.org/wiki/SpaceX", "technology"),
    ("https://en.wikipedia.org/wiki/Tesla,_Inc.", "technology"),
    ("https://en.wikipedia.org/wiki/Apple_Inc.", "technology"),
    ("https://en.wikipedia.org/wiki/Google", "technology"),
    ("https://en.wikipedia.org/wiki/Microsoft", "technology"),
    ("https://en.wikipedia.org/wiki/Amazon_(company)", "technology"),
    ("https://en.wikipedia.org/wiki/Facebook", "technology"),
    
    # ===== SCIENCE (25 URLs) =====
    ("https://en.wikipedia.org/wiki/Climate_change", "science"),
    ("https://en.wikipedia.org/wiki/DNA", "science"),
    ("https://en.wikipedia.org/wiki/Evolution", "science"),
    ("https://en.wikipedia.org/wiki/Black_hole", "science"),
    ("https://en.wikipedia.org/wiki/Big_Bang", "science"),
    ("https://en.wikipedia.org/wiki/Quantum_mechanics", "science"),
    ("https://en.wikipedia.org/wiki/Theory_of_relativity", "science"),
    ("https://en.wikipedia.org/wiki/Photosynthesis", "science"),
    ("https://en.wikipedia.org/wiki/Human_brain", "science"),
    ("https://en.wikipedia.org/wiki/Solar_System", "science"),
    ("https://en.wikipedia.org/wiki/Milky_Way", "science"),
    ("https://en.wikipedia.org/wiki/Periodic_table", "science"),
    ("https://en.wikipedia.org/wiki/Atom", "science"),
    ("https://en.wikipedia.org/wiki/Cell_(biology)", "science"),
    ("https://en.wikipedia.org/wiki/Genetics", "science"),
    ("https://en.wikipedia.org/wiki/Vaccination", "science"),
    ("https://en.wikipedia.org/wiki/Antibiotic", "science"),
    ("https://en.wikipedia.org/wiki/Nuclear_power", "science"),
    ("https://en.wikipedia.org/wiki/Renewable_energy", "science"),
    ("https://en.wikipedia.org/wiki/Global_warming", "science"),
    ("https://en.wikipedia.org/wiki/Biodiversity", "science"),
    ("https://en.wikipedia.org/wiki/Ecosystem", "science"),
    ("https://en.wikipedia.org/wiki/Oceanography", "science"),
    ("https://en.wikipedia.org/wiki/Astronomy", "science"),
    ("https://en.wikipedia.org/wiki/Neuroscience", "science"),
    
    # ===== ARTS (20 URLs) =====
    ("https://en.wikipedia.org/wiki/Mona_Lisa", "arts"),
    ("https://en.wikipedia.org/wiki/Starry_Night", "arts"),
    ("https://en.wikipedia.org/wiki/The_Last_Supper_(Leonardo)", "arts"),
    ("https://en.wikipedia.org/wiki/Sistine_Chapel_ceiling", "arts"),
    ("https://en.wikipedia.org/wiki/The_Scream", "arts"),
    ("https://en.wikipedia.org/wiki/Girl_with_a_Pearl_Earring", "arts"),
    ("https://en.wikipedia.org/wiki/The_Birth_of_Venus", "arts"),
    ("https://en.wikipedia.org/wiki/Guernica_(Picasso)", "arts"),
    ("https://en.wikipedia.org/wiki/The_Persistence_of_Memory", "arts"),
    ("https://en.wikipedia.org/wiki/The_Great_Wave_off_Kanagawa", "arts"),
    ("https://en.wikipedia.org/wiki/Statue_of_Liberty", "arts"),
    ("https://en.wikipedia.org/wiki/Eiffel_Tower", "arts"),
    ("https://en.wikipedia.org/wiki/Great_Wall_of_China", "arts"),
    ("https://en.wikipedia.org/wiki/Taj_Mahal", "arts"),
    ("https://en.wikipedia.org/wiki/Colosseum", "arts"),
    ("https://en.wikipedia.org/wiki/Machu_Picchu", "arts"),
    ("https://en.wikipedia.org/wiki/Petra", "arts"),
    ("https://en.wikipedia.org/wiki/Louvre", "arts"),
    ("https://en.wikipedia.org/wiki/Metropolitan_Museum_of_Art", "arts"),
    ("https://en.wikipedia.org/wiki/British_Museum", "arts"),
    
    # ===== SPORTS (15 URLs) =====
    ("https://en.wikipedia.org/wiki/Olympic_Games", "sports"),
    ("https://en.wikipedia.org/wiki/FIFA_World_Cup", "sports"),
    ("https://en.wikipedia.org/wiki/Super_Bowl", "sports"),
    ("https://en.wikipedia.org/wiki/NBA", "sports"),
    ("https://en.wikipedia.org/wiki/UEFA_Champions_League", "sports"),
    ("https://en.wikipedia.org/wiki/Wimbledon_Championships", "sports"),
    ("https://en.wikipedia.org/wiki/Tour_de_France", "sports"),
    ("https://en.wikipedia.org/wiki/Formula_One", "sports"),
    ("https://en.wikipedia.org/wiki/Cricket_World_Cup", "sports"),
    ("https://en.wikipedia.org/wiki/Rugby_World_Cup", "sports"),
    ("https://en.wikipedia.org/wiki/Association_football", "sports"),
    ("https://en.wikipedia.org/wiki/Basketball", "sports"),
    ("https://en.wikipedia.org/wiki/Tennis", "sports"),
    ("https://en.wikipedia.org/wiki/Golf", "sports"),
    ("https://en.wikipedia.org/wiki/Swimming_(sport)", "sports"),
    
    # ===== ORGANIZATION (10 URLs) =====
    ("https://en.wikipedia.org/wiki/United_Nations", "organization"),
    ("https://en.wikipedia.org/wiki/World_Health_Organization", "organization"),
    ("https://en.wikipedia.org/wiki/NASA", "organization"),
    ("https://en.wikipedia.org/wiki/European_Union", "organization"),
    ("https://en.wikipedia.org/wiki/NATO", "organization"),
    ("https://en.wikipedia.org/wiki/Red_Cross", "organization"),
    ("https://en.wikipedia.org/wiki/Greenpeace", "organization"),
    ("https://en.wikipedia.org/wiki/Amnesty_International", "organization"),
    ("https://en.wikipedia.org/wiki/World_Bank", "organization"),
    ("https://en.wikipedia.org/wiki/International_Monetary_Fund", "organization"),
]

print(f"Defined {len(fixed_urls)} fixed Wikipedia URLs")

# Count URLs per domain
domain_counts = {}
for url, domain in fixed_urls:
    domain_counts[domain] = domain_counts.get(domain, 0) + 1

print("\nDistribution by domain:")
for domain, count in sorted(domain_counts.items(), key=lambda x: -x[1]):
    print(f"  - {domain}: {count} URLs")

Defined 200 fixed Wikipedia URLs

Distribution by domain:
  - people: 50 URLs
  - places: 30 URLs
  - events: 25 URLs
  - technology: 25 URLs
  - science: 25 URLs
  - arts: 20 URLs
  - sports: 15 URLs
  - organization: 10 URLs


## Create Fixed URL Dataset

This cell processes all 200 fixed Wikipedia URLs and creates the dataset.
For each URL, it:
1. Creates a dataset entry with unique ID, URL, domain, and content
2. Extracts up to 200 words from the Wikipedia page
3. Saves the complete dataset to `fixed_url.json`

**Note:** Processing 200 URLs will take approximately 5-10 minutes due to respectful rate limiting.

In [9]:
import os

def create_fixed_url_dataset():
    """
    Creates the fixed URL dataset from predefined Wikipedia URLs.
    
    Returns:
        list: List of dataset entries
    """
    print("Creating Fixed URL Dataset...")
    print("=" * 50)
    
    dataset = []
    total_urls = len(fixed_urls)
    
    for i, (url, domain) in enumerate(fixed_urls, start=1):
        print(f"[{i}/{total_urls}] Processing: {url.split('/')[-1].replace('_', ' ')}")
        entry = create_dataset_entry(url, domain)
        dataset.append(entry)
        print(f"  ✓ Added: {url.split('/')[-1].replace('_', ' ')} ({domain})")
    
    return dataset

# Check if fixed_url.json exists, if so load it, otherwise create it

if os.path.exists("fixed_url.json"):
    print("fixed_url.json already exists. Loading from file...")
    with open('fixed_url.json', 'r', encoding='utf-8') as f:
        fixed_dataset = json.load(f)
    print(f"Loaded {len(fixed_dataset)} entries from fixed_url.json")
else:
    fixed_dataset = create_fixed_url_dataset()
    save_to_json(fixed_dataset, "fixed_url.json")

Creating Fixed URL Dataset...
[1/200] Processing: Albert Einstein
Processing: https://en.wikipedia.org/wiki/Albert_Einstein
  ✓ Added: Albert Einstein (people)
[2/200] Processing: Marie Curie
Processing: https://en.wikipedia.org/wiki/Marie_Curie
  ✓ Added: Marie Curie (people)
[3/200] Processing: Isaac Newton
Processing: https://en.wikipedia.org/wiki/Isaac_Newton
  ✓ Added: Isaac Newton (people)
[4/200] Processing: Charles Darwin
Processing: https://en.wikipedia.org/wiki/Charles_Darwin
  ✓ Added: Charles Darwin (people)
[5/200] Processing: Nikola Tesla
Processing: https://en.wikipedia.org/wiki/Nikola_Tesla
  ✓ Added: Nikola Tesla (people)
[6/200] Processing: Stephen Hawking
Processing: https://en.wikipedia.org/wiki/Stephen_Hawking
  ✓ Added: Stephen Hawking (people)
[7/200] Processing: Galileo Galilei
Processing: https://en.wikipedia.org/wiki/Galileo_Galilei
  ✓ Added: Galileo Galilei (people)
[8/200] Processing: Richard Feynman
Processing: https://en.wikipedia.org/wiki/Richard_Feynman

## Create Random URL Dataset

This cell generates a dataset from 300 randomly selected Wikipedia articles.
For each random article, it:
1. Fetches a random Wikipedia article URL
2. Automatically detects the domain/category of the article
3. Creates a dataset entry with unique ID, URL, detected domain, and content
4. Saves the complete dataset to `random_url.json`

**Note:** This cell **always recreates** `random_url.json` with fresh random URLs (overwrites existing file).
Processing 300 random URLs will take approximately 15-20 minutes due to respectful rate limiting.

In [10]:
def create_random_url_dataset(count=300):
    """
    Creates a dataset from randomly selected Wikipedia articles.
    
    Args:
        count (int): Number of random articles to fetch (default: 300)
    
    Returns:
        list: List of dataset entries
    """
    print(f"\nCreating Random URL Dataset ({count} articles)...")
    print("=" * 50)
    
    dataset = []
    
    for i in range(1, count + 1):
        print(f"\n[{i}/{count}] Fetching random article...")
        
        url = get_random_wikipedia_url()
        
        if url:
            # Detect the domain of the random article
            domain = detect_domain(url)
            entry = create_dataset_entry(url, domain)
            dataset.append(entry)
            
            article_name = url.split('/')[-1].replace('_', ' ')
            print(f"  ✓ Added: {article_name} ({domain})")
        else:
            print(f"  ✗ Failed to fetch random article")
        
        # Small delay between requests
        time.sleep(1)
    
    return dataset

# Always recreate random_url.json with 300 random URLs (overwrites if exists)
print("Creating random_url.json with 300 random Wikipedia URLs...")
print("(This will overwrite any existing file)")
random_dataset = create_random_url_dataset(300)
save_to_json(random_dataset, "random_url.json")

Creating random_url.json with 300 random Wikipedia URLs...
(This will overwrite any existing file)

Creating Random URL Dataset (300 articles)...

[1/300] Fetching random article...
Error getting random URL: HTTPSConnectionPool(host='en.wikipedia.org', port=443): Max retries exceeded with url: /wiki/Special:Random (Caused by ConnectTimeoutError(<HTTPSConnection(host='en.wikipedia.org', port=443) at 0x1d763444b90>, 'Connection to en.wikipedia.org timed out. (connect timeout=10)'))
  ✗ Failed to fetch random article

[2/300] Fetching random article...
Processing: https://en.wikipedia.org/wiki/Zenepos_totolirata
  ✓ Added: Zenepos totolirata (other)

[3/300] Fetching random article...
Processing: https://en.wikipedia.org/wiki/Taxation_in_Massachusetts
  ✓ Added: Taxation in Massachusetts (other)

[4/300] Fetching random article...
Processing: https://en.wikipedia.org/wiki/Naunihal
  ✓ Added: Naunihal (arts)

[5/300] Fetching random article...
Processing: https://en.wikipedia.org/wiki/Kris

## Display Dataset Summary

This cell provides a summary of both created datasets, including:
- Total number of entries in each dataset
- Sample entries from each dataset
- Domain distribution statistics

In [11]:
def display_dataset_summary(dataset, name):
    """
    Displays a summary of a dataset.
    
    Args:
        dataset (list): The dataset to summarize
        name (str): Name of the dataset
    """
    print(f"\n{'=' * 60}")
    print(f"Dataset Summary: {name}")
    print(f"{'=' * 60}")
    print(f"Total Entries: {len(dataset)}")
    
    # Count domains
    domain_counts = {}
    for entry in dataset:
        domain = entry['domain']
        domain_counts[domain] = domain_counts.get(domain, 0) + 1
    
    print(f"\nDomain Distribution:")
    for domain, count in sorted(domain_counts.items()):
        print(f"  - {domain}: {count}")
    
    print(f"\nSample Entry:")
    if dataset:
        sample = dataset[0]
        print(f"  ID: {sample['id']}")
        print(f"  URL: {sample['url']}")
        print(f"  Domain: {sample['domain']}")
        content_preview = sample['content'][:150] + "..." if len(sample['content']) > 150 else sample['content']
        print(f"  Content Preview: {content_preview}")

# Display summaries
display_dataset_summary(fixed_dataset, "fixed_url.json")
display_dataset_summary(random_dataset, "random_url.json")


Dataset Summary: fixed_url.json
Total Entries: 200

Domain Distribution:
  - arts: 20
  - events: 25
  - organization: 10
  - people: 50
  - places: 30
  - science: 25
  - sports: 15
  - technology: 25

Sample Entry:
  ID: a79f6efb-ca0d-46a4-9a87-749c3e9084be
  URL: https://en.wikipedia.org/wiki/Albert_Einstein
  Domain: people
  Content Preview: Albert Einstein[a] (14 March 1879 – 18 April 1955) was a German-born theoretical physicist best known for developing the theory of relativity. Einstei...

Dataset Summary: random_url.json
Total Entries: 297

Domain Distribution:
  - arts: 37
  - events: 19
  - organization: 3
  - other: 55
  - people: 98
  - places: 58
  - science: 4
  - sports: 21
  - technology: 2

Sample Entry:
  ID: 0e7f42aa-7a33-4e9e-828f-f03d23d88366
  URL: https://en.wikipedia.org/wiki/Zenepos_totolirata
  Domain: other
  Content Preview: Daphnella totolirata Suter, 1908 Zenepos totolirata is a species of sea snail, a marine gastropod mollusk in the family Raphitomidae

## Verify JSON Files

This cell verifies that both JSON files were created correctly by:
1. Reading the files back from disk
2. Validating the JSON structure
3. Confirming the number of entries in each file

In [12]:
def verify_json_files():
    """
    Verifies that the JSON files were created correctly.
    """
    print("\nVerifying JSON Files...")
    print("=" * 50)
    
    files_to_verify = ["fixed_url.json", "random_url.json"]
    
    for filename in files_to_verify:
        try:
            with open(filename, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            print(f"\n✓ {filename}:")
            print(f"  - File exists and is valid JSON")
            print(f"  - Contains {len(data)} entries")
            
            # Verify structure of first entry
            if data:
                first_entry = data[0]
                required_fields = ['id', 'url', 'domain', 'content']
                has_all_fields = all(field in first_entry for field in required_fields)
                print(f"  - All required fields present: {has_all_fields}")
                
        except FileNotFoundError:
            print(f"\n✗ {filename}: File not found")
        except json.JSONDecodeError:
            print(f"\n✗ {filename}: Invalid JSON format")

verify_json_files()


Verifying JSON Files...

✓ fixed_url.json:
  - File exists and is valid JSON
  - Contains 200 entries
  - All required fields present: True

✓ random_url.json:
  - File exists and is valid JSON
  - Contains 297 entries
  - All required fields present: True


## Conclusion

This notebook has successfully created two JSON datasets:

1. **fixed_url.json**: Contains 200 entries from predefined Wikipedia URLs covering 8 domains:
   - People (50), Places (30), Events (25), Technology (25), Science (25), Arts (20), Sports (15), Organization (10)

2. **random_url.json**: Contains 300 entries from randomly selected Wikipedia articles with automatically detected domains.

Each entry in both datasets contains:
- A unique GUID identifier
- The Wikipedia page URL
- The domain/category of the page
- Up to 200 words of content extracted from the page

# 1.1 Dense Vector Retrieval

This section implements Dense Vector Retrieval using sentence embeddings. The approach involves:
1. **Sentence Embedding Model**: Using `all-MiniLM-L6-v2` (or `all-mpnet-base-v2`) from the sentence-transformers library to convert text chunks into dense vector representations
2. **Vector Index**: Building a FAISS index for efficient similarity search
3. **Retrieval**: Finding top-K most similar chunks using cosine similarity

## Import Libraries for Dense Vector Retrieval

This cell imports the required libraries for dense vector retrieval:
- `sentence_transformers`: For loading pre-trained sentence embedding models
- `faiss`: Facebook AI Similarity Search library for efficient vector indexing and retrieval
- `numpy`: For numerical operations on embeddings

In [13]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import json

c:\Users\koush\OneDrive\Documents\BITS Pilani\Conversational AI\Assignment 2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Datasets from JSON Files

This cell loads the previously created JSON datasets (`fixed_url.json` and `random_url.json`) and combines them into a single corpus for vector retrieval. Each document contains the content, URL, domain, and unique ID.

In [14]:
def load_datasets():
    """
    Loads both fixed and random URL datasets from JSON files.
    
    Returns:
        list: Combined list of all document entries from both datasets
    """
    all_documents = []
    
    # Load fixed URL dataset
    with open('fixed_url.json', 'r', encoding='utf-8') as f:
        fixed_data = json.load(f)
        all_documents.extend(fixed_data)
        print(f"Loaded {len(fixed_data)} documents from fixed_url.json")
    
    # Load random URL dataset
    with open('random_url.json', 'r', encoding='utf-8') as f:
        random_data = json.load(f)
        all_documents.extend(random_data)
        print(f"Loaded {len(random_data)} documents from random_url.json")
    
    print(f"\nTotal documents in corpus: {len(all_documents)}")
    return all_documents

# Load the datasets
corpus = load_datasets()

Loaded 200 documents from fixed_url.json
Loaded 297 documents from random_url.json

Total documents in corpus: 497


## Initialize Sentence Transformer Model

This cell initializes the sentence embedding model. We use `all-MiniLM-L6-v2` which is:
- A lightweight model (80MB) with 384-dimensional embeddings
- Fast inference speed while maintaining good quality
- Trained on a large corpus for semantic similarity tasks

Alternative: `all-mpnet-base-v2` provides higher quality embeddings (768 dimensions) but is larger and slower.

In [15]:
# Install required packages including certifi for SSL certificates
%pip install sentence-transformers certifi --upgrade -q

# Fix SSL certificate path issue
import certifi
import os
os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()
print(f"SSL certificates configured: {certifi.where()}")

Note: you may need to restart the kernel to use updated packages.
SSL certificates configured: c:\Users\koush\OneDrive\Documents\BITS Pilani\Conversational AI\Assignment 2\.venv\Lib\site-packages\certifi\cacert.pem


In [16]:
# Fix SSL certificate issue BEFORE importing sentence_transformers
import ssl
import certifi
import os

# Set environment variables for SSL - must be done before any HTTPS requests
os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()
os.environ['CURL_CA_BUNDLE'] = certifi.where()

# Verify certificate file exists
cert_path = certifi.where()
if os.path.exists(cert_path):
    print(f"✓ SSL certificate file found: {cert_path}")
else:
    raise FileNotFoundError(f"SSL certificate file not found at {cert_path}. Run: pip install --force-reinstall certifi")

# Now import and initialize the sentence transformer model
from sentence_transformers import SentenceTransformer

MODEL_NAME = 'all-MiniLM-L6-v2'
# MODEL_NAME = 'all-mpnet-base-v2'  # Uncomment for higher quality embeddings

print(f"\nLoading sentence transformer model: {MODEL_NAME}")
embedding_model = SentenceTransformer(MODEL_NAME)
print(f"Model loaded successfully!")
print(f"Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")

✓ SSL certificate file found: c:\Users\koush\OneDrive\Documents\BITS Pilani\Conversational AI\Assignment 2\.venv\Lib\site-packages\certifi\cacert.pem

Loading sentence transformer model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 563.14it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully!
Embedding dimension: 384


## Generate Embeddings for Document Corpus

This cell creates dense vector embeddings for all documents in our corpus. The function:
1. Extracts the content text from each document
2. Uses the sentence transformer model to encode all texts into vectors
3. Normalizes the embeddings for cosine similarity computation
4. Returns the embeddings as a numpy array suitable for FAISS indexing

In [17]:
def generate_embeddings(documents, model):
    """
    Generates dense vector embeddings for all documents in the corpus.
    
    Args:
        documents (list): List of document dictionaries containing 'content' field
        model (SentenceTransformer): The sentence transformer model for encoding
    
    Returns:
        numpy.ndarray: Matrix of document embeddings (num_docs x embedding_dim)
    """
    # Extract content from documents
    texts = [doc['content'] for doc in documents]
    
    print(f"Generating embeddings for {len(texts)} documents...")
    
    # Generate embeddings
    embeddings = model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
    
    # Normalize embeddings for cosine similarity (FAISS inner product on normalized vectors = cosine similarity)
    faiss.normalize_L2(embeddings)
    
    print(f"Embeddings shape: {embeddings.shape}")
    return embeddings

# Generate embeddings for the corpus
corpus_embeddings = generate_embeddings(corpus, embedding_model)

Generating embeddings for 497 documents...


Batches: 100%|██████████| 16/16 [00:14<00:00,  1.08it/s]

Embeddings shape: (497, 384)


## Build FAISS Vector Index

This cell creates a FAISS index for efficient similarity search. We use:
- `IndexFlatIP` (Inner Product): Since embeddings are L2-normalized, inner product equals cosine similarity
- This is an exact search index that provides 100% recall

For larger datasets, approximate indices like `IndexIVFFlat` or `IndexHNSW` can be used for faster search with slight accuracy trade-off.

In [18]:
def build_faiss_index(embeddings):
    """
    Builds a FAISS index for efficient similarity search.
    
    Args:
        embeddings (numpy.ndarray): Matrix of document embeddings
    
    Returns:
        faiss.Index: FAISS index containing all document embeddings
    """
    embedding_dim = embeddings.shape[1]
    
    # Create a FAISS index using Inner Product (cosine similarity for normalized vectors)
    index = faiss.IndexFlatIP(embedding_dim)
    
    # Add embeddings to the index
    index.add(embeddings)
    
    print(f"FAISS index built successfully!")
    print(f"Index type: IndexFlatIP (Inner Product / Cosine Similarity)")
    print(f"Embedding dimension: {embedding_dim}")
    print(f"Total vectors in index: {index.ntotal}")
    
    return index

# Build the FAISS index
faiss_index = build_faiss_index(corpus_embeddings)

FAISS index built successfully!
Index type: IndexFlatIP (Inner Product / Cosine Similarity)
Embedding dimension: 384
Total vectors in index: 497


## Dense Vector Retrieval Function

This cell implements the core retrieval function that:
1. Takes a query string as input
2. Encodes the query using the same sentence transformer model
3. Normalizes the query embedding for cosine similarity
4. Searches the FAISS index to find the top-K most similar documents
5. Returns the matching documents with their similarity scores

In [19]:
def retrieve_top_k(query, model, index, documents, k=5):
    """
    Retrieves the top-K most similar documents for a given query using dense vector search.
    
    Args:
        query (str): The search query string
        model (SentenceTransformer): The sentence transformer model for encoding
        index (faiss.Index): The FAISS index containing document embeddings
        documents (list): List of original document dictionaries
        k (int): Number of top results to retrieve (default: 5)
    
    Returns:
        list: List of tuples containing (document, similarity_score)
    """
    # Encode the query
    query_embedding = model.encode([query], convert_to_numpy=True)
    
    # Normalize for cosine similarity
    faiss.normalize_L2(query_embedding)
    
    # Search the index
    similarities, indices = index.search(query_embedding, k)
    
    # Collect results
    results = []
    for i, (sim, idx) in enumerate(zip(similarities[0], indices[0])):
        if idx < len(documents):  # Valid index check
            results.append({
                'rank': i + 1,
                'similarity_score': float(sim),
                'document': documents[idx]
            })
    
    return results

def display_retrieval_results(query, results):
    """
    Displays the retrieval results in a formatted manner.
    
    Args:
        query (str): The search query
        results (list): List of retrieval results
    """
    print(f"\n{'=' * 70}")
    print(f"Query: \"{query}\"")
    print(f"{'=' * 70}")
    
    for result in results:
        print(f"\n[Rank {result['rank']}] Similarity: {result['similarity_score']:.4f}")
        doc = result['document']
        print(f"  URL: {doc['url']}")
        print(f"  Domain: {doc['domain']}")
        content_preview = doc['content'][:200] + "..." if len(doc['content']) > 200 else doc['content']
        print(f"  Content: {content_preview}")

## Test Dense Vector Retrieval

This cell demonstrates the dense vector retrieval system with several example queries. Each query is encoded and compared against all documents in the corpus using cosine similarity. The top-K most relevant documents are returned based on semantic similarity rather than keyword matching.

In [20]:
# Test queries for dense vector retrieval
test_queries = [
    "Where is Mona Lisa located?",
    "Tell me about a natural wonder in the United States",
    "What was a major conflict in the 20th century?",
    "Information about artificial intelligence and machine learning",
    "Famous artwork from the Renaissance period"
]

# Perform retrieval for each query
print("DENSE VECTOR RETRIEVAL DEMONSTRATION")
print("=" * 70)
print(f"Model: {MODEL_NAME}")
print(f"Corpus size: {len(corpus)} documents")
print(f"Retrieving top-3 results for each query")

for query in test_queries:
    results = retrieve_top_k(query, embedding_model, faiss_index, corpus, k=3)
    display_retrieval_results(query, results)

DENSE VECTOR RETRIEVAL DEMONSTRATION
Model: all-MiniLM-L6-v2
Corpus size: 497 documents
Retrieving top-3 results for each query

Query: "Where is Mona Lisa located?"

[Rank 1] Similarity: 0.5913
  URL: https://en.wikipedia.org/wiki/Mona_Lisa
  Domain: arts
  Content: The Mona Lisa[a] is a half-length portrait painting by the Italian artist Leonardo da Vinci. Considered an archetypal masterpiece of the Italian Renaissance, it has been described as "the best known, ...

[Rank 2] Similarity: 0.5340
  URL: https://en.wikipedia.org/wiki/Louvre
  Domain: arts
  Content: The Louvre,[a] or the Louvre Museum (French: Musée du Louvre [myze dy luvʁ] ⓘ), is a national art museum in Paris, France. The Louvre, a former royal palace, is known for its collection of celebrated ...

[Rank 3] Similarity: 0.3673
  URL: https://en.wikipedia.org/wiki/The_Last_Supper_(Leonardo)
  Domain: arts
  Content: The Last Supper (Italian: Il Cenacolo [il tʃeˈnaːkolo] or L'Ultima Cena [ˈlultima ˈtʃeːna]) is a mural pai

## Summary: Dense Vector Retrieval

This section implemented a complete Dense Vector Retrieval system:

1. **Embedding Model**: Used `all-MiniLM-L6-v2` sentence transformer to convert text into 384-dimensional dense vectors
2. **Vector Index**: Built a FAISS `IndexFlatIP` index for exact cosine similarity search
3. **Retrieval**: Implemented top-K retrieval based on semantic similarity between query and document embeddings

**Key Advantages of Dense Vector Retrieval:**
- Captures semantic meaning, not just keyword matches
- Works well with synonyms and paraphrased queries
- Efficient similarity search using optimized vector operations

# 1.2 Sparse Keyword Retrieval (BM25)

This section implements Sparse Keyword Retrieval using the BM25 (Best Matching 25) algorithm. The approach involves:
1. **Text Preprocessing**: Tokenizing documents into words and removing stopwords
2. **BM25 Index**: Building a BM25 index over the document corpus
3. **Keyword-based Retrieval**: Finding top-K documents based on term frequency and inverse document frequency

**BM25 Formula:**
$$\text{score}(D, Q) = \sum_{i=1}^{n} \text{IDF}(q_i) \cdot \frac{f(q_i, D) \cdot (k_1 + 1)}{f(q_i, D) + k_1 \cdot (1 - b + b \cdot \frac{|D|}{\text{avgdl}})}$$

Where:
- $f(q_i, D)$ = frequency of term $q_i$ in document $D$
- $|D|$ = length of document $D$
- $\text{avgdl}$ = average document length
- $k_1$ and $b$ = tuning parameters (typically $k_1 = 1.5$, $b = 0.75$)

## Import Libraries for BM25 Retrieval

This cell imports the required libraries for BM25-based sparse retrieval:
- `rank_bm25`: Implementation of the BM25 ranking algorithm
- `nltk`: Natural Language Toolkit for text preprocessing (tokenization, stopwords)
- `string`: For punctuation handling

In [21]:
from rank_bm25 import BM25Okapi
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string

# Download required NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("BM25 libraries imported successfully!")

BM25 libraries imported successfully!


## Text Preprocessing Function

This cell defines the text preprocessing function for BM25. The function:
1. Converts text to lowercase for case-insensitive matching
2. Removes punctuation marks
3. Tokenizes text into individual words
4. Removes common English stopwords (the, is, at, etc.)
5. Returns a list of cleaned tokens ready for BM25 indexing

In [22]:
def preprocess_text_for_bm25(text):
    """
    Preprocesses text for BM25 indexing by tokenizing and removing stopwords.
    
    Args:
        text (str): The input text to preprocess
    
    Returns:
        list: List of preprocessed tokens
    """
    # Convert to lowercase
    text = text.lower()
    
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [token for token in tokens if token not in stop_words and len(token) > 1]
    
    return tokens

# Test the preprocessing function
sample_text = "The quick brown fox jumps over the lazy dog!"
processed = preprocess_text_for_bm25(sample_text)
print(f"Original: {sample_text}")
print(f"Processed tokens: {processed}")

Original: The quick brown fox jumps over the lazy dog!
Processed tokens: ['quick', 'brown', 'fox', 'jumps', 'lazy', 'dog']


## Build BM25 Index

This cell builds the BM25 index over the document corpus. The function:
1. Extracts content from each document in the corpus
2. Preprocesses each document using tokenization and stopword removal
3. Creates a BM25Okapi index from the tokenized documents
4. Returns the index along with the tokenized corpus for later use

In [23]:
def build_bm25_index(documents):
    """
    Builds a BM25 index over the document corpus.
    
    Args:
        documents (list): List of document dictionaries containing 'content' field
    
    Returns:
        tuple: (BM25Okapi index, list of tokenized documents)
    """
    print("Building BM25 index...")
    
    # Tokenize all documents
    tokenized_corpus = []
    for doc in documents:
        tokens = preprocess_text_for_bm25(doc['content'])
        tokenized_corpus.append(tokens)
    
    # Create BM25 index
    bm25_index = BM25Okapi(tokenized_corpus)
    
    # Calculate statistics
    total_tokens = sum(len(tokens) for tokens in tokenized_corpus)
    avg_doc_length = total_tokens / len(tokenized_corpus)
    
    print(f"BM25 index built successfully!")
    print(f"Total documents indexed: {len(tokenized_corpus)}")
    print(f"Total tokens: {total_tokens}")
    print(f"Average document length: {avg_doc_length:.2f} tokens")
    
    return bm25_index, tokenized_corpus

# Build the BM25 index using the same corpus from dense retrieval
bm25_index, tokenized_corpus = build_bm25_index(corpus)

Building BM25 index...
BM25 index built successfully!
Total documents indexed: 497
Total tokens: 49422
Average document length: 99.44 tokens


## BM25 Retrieval Function

This cell implements the BM25 retrieval function that:
1. Takes a query string as input
2. Preprocesses the query using the same tokenization pipeline
3. Computes BM25 scores for all documents in the corpus
4. Returns the top-K documents sorted by their BM25 scores

In [24]:
def retrieve_top_k_bm25(query, bm25_index, documents, k=5):
    """
    Retrieves the top-K most relevant documents for a given query using BM25.
    
    Args:
        query (str): The search query string
        bm25_index (BM25Okapi): The BM25 index
        documents (list): List of original document dictionaries
        k (int): Number of top results to retrieve (default: 5)
    
    Returns:
        list: List of dictionaries containing rank, score, and document
    """
    # Preprocess the query
    query_tokens = preprocess_text_for_bm25(query)
    
    # Get BM25 scores for all documents
    scores = bm25_index.get_scores(query_tokens)
    
    # Get top-K document indices
    top_k_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    
    # Collect results
    results = []
    for rank, idx in enumerate(top_k_indices, 1):
        results.append({
            'rank': rank,
            'bm25_score': float(scores[idx]),
            'document': documents[idx]
        })
    
    return results

def display_bm25_results(query, results):
    """
    Displays the BM25 retrieval results in a formatted manner.
    
    Args:
        query (str): The search query
        results (list): List of retrieval results
    """
    print(f"\n{'=' * 70}")
    print(f"Query: \"{query}\"")
    print(f"Query tokens: {preprocess_text_for_bm25(query)}")
    print(f"{'=' * 70}")
    
    for result in results:
        print(f"\n[Rank {result['rank']}] BM25 Score: {result['bm25_score']:.4f}")
        doc = result['document']
        print(f"  URL: {doc['url']}")
        print(f"  Domain: {doc['domain']}")
        content_preview = doc['content'][:200] + "..." if len(doc['content']) > 200 else doc['content']
        print(f"  Content: {content_preview}")

## Test BM25 Sparse Retrieval

This cell demonstrates the BM25 keyword-based retrieval system with several example queries. Unlike dense vector retrieval which captures semantic meaning, BM25 relies on exact keyword matching with term frequency weighting.

In [25]:
# Test queries for BM25 retrieval
bm25_test_queries = [
    "Einstein physics relativity theory",
    "Grand Canyon natural wonder Arizona",
    "World War military conflict",
    "artificial intelligence machine learning",
    "Mona Lisa Leonardo painting"
]

# Perform BM25 retrieval for each query
print("BM25 SPARSE KEYWORD RETRIEVAL DEMONSTRATION")
print("=" * 70)
print(f"Corpus size: {len(corpus)} documents")
print(f"Retrieving top-3 results for each query")

for query in bm25_test_queries:
    results = retrieve_top_k_bm25(query, bm25_index, corpus, k=3)
    display_bm25_results(query, results)

BM25 SPARSE KEYWORD RETRIEVAL DEMONSTRATION
Corpus size: 497 documents
Retrieving top-3 results for each query

Query: "Einstein physics relativity theory"
Query tokens: ['einstein', 'physics', 'relativity', 'theory']

[Rank 1] BM25 Score: 28.1761
  URL: https://en.wikipedia.org/wiki/Theory_of_relativity
  Domain: science
  Content: The theory of relativity comprises two physics theories by Albert Einstein: special relativity and general relativity, proposed and published in 1905 and 1915, respectively. Special relativity applies...

[Rank 2] BM25 Score: 23.0975
  URL: https://en.wikipedia.org/wiki/Albert_Einstein
  Domain: people
  Content: Albert Einstein[a] (14 March 1879 – 18 April 1955) was a German-born theoretical physicist best known for developing the theory of relativity. Einstein also made important contributions to quantum the...

[Rank 3] BM25 Score: 11.8210
  URL: https://en.wikipedia.org/wiki/Black_hole
  Domain: science
  Content: A black hole is an astronomical body so

## Compare Dense vs Sparse Retrieval

This cell compares the results from Dense Vector Retrieval (semantic) and Sparse BM25 Retrieval (keyword-based) for the same queries. This helps illustrate the differences between the two approaches:
- **Dense Retrieval**: Better at understanding meaning and context
- **Sparse Retrieval**: Better at exact keyword matching

In [26]:
def compare_retrieval_methods(query, k=3):
    """
    Compares Dense Vector and BM25 retrieval results for the same query.
    
    Args:
        query (str): The search query
        k (int): Number of top results to retrieve
    """
    print(f"\n{'#' * 80}")
    print(f"COMPARISON: \"{query}\"")
    print(f"{'#' * 80}")
    
    # Dense Vector Retrieval
    print(f"\n--- DENSE VECTOR RETRIEVAL (Semantic) ---")
    dense_results = retrieve_top_k(query, embedding_model, faiss_index, corpus, k)
    for result in dense_results:
        doc = result['document']
        title = doc['url'].split('/')[-1].replace('_', ' ')
        print(f"  [{result['rank']}] {title} (Score: {result['similarity_score']:.4f})")
    
    # BM25 Retrieval
    print(f"\n--- BM25 SPARSE RETRIEVAL (Keyword) ---")
    bm25_results = retrieve_top_k_bm25(query, bm25_index, corpus, k)
    for result in bm25_results:
        doc = result['document']
        title = doc['url'].split('/')[-1].replace('_', ' ')
        print(f"  [{result['rank']}] {title} (Score: {result['bm25_score']:.4f})")

# Compare both methods with test queries
comparison_queries = [
    "famous scientist who developed the theory of relativity",
    "Grand Canyon natural wonder",
    "painting by Leonardo da Vinci"
]

print("RETRIEVAL METHOD COMPARISON")
print("=" * 80)
for query in comparison_queries:
    compare_retrieval_methods(query, k=3)

RETRIEVAL METHOD COMPARISON

################################################################################
COMPARISON: "famous scientist who developed the theory of relativity"
################################################################################

--- DENSE VECTOR RETRIEVAL (Semantic) ---
  [1] Theory of relativity (Score: 0.7070)
  [2] Isaac Newton (Score: 0.6333)
  [3] Albert Einstein (Score: 0.5991)

--- BM25 SPARSE RETRIEVAL (Keyword) ---
  [1] Theory of relativity (Score: 17.4952)
  [2] Albert Einstein (Score: 13.9079)
  [3] Black hole (Score: 11.8210)

################################################################################
COMPARISON: "Grand Canyon natural wonder"
################################################################################

--- DENSE VECTOR RETRIEVAL (Semantic) ---
  [1] Grand Canyon (Score: 0.6178)
  [2] Yellowstone National Park (Score: 0.4272)
  [3] Niagara Falls (Score: 0.3045)

--- BM25 SPARSE RETRIEVAL (Keyword) ---
  [1] Grand Ca

## Summary: BM25 Sparse Keyword Retrieval

This section implemented a complete BM25 Sparse Keyword Retrieval system:

1. **Text Preprocessing**: Tokenization, lowercase conversion, punctuation removal, and stopword filtering
2. **BM25 Index**: Built using the BM25Okapi algorithm from rank_bm25 library
3. **Retrieval**: Keyword-based retrieval using term frequency and inverse document frequency scoring

**Key Characteristics of BM25:**
- Based on term frequency (TF) and inverse document frequency (IDF)
- Considers document length normalization
- Excels at exact keyword matching
- Fast indexing and retrieval for large corpora

**Comparison with Dense Retrieval:**
| Aspect | Dense Vector | BM25 Sparse |
|--------|--------------|-------------|
| Matching | Semantic similarity | Keyword matching |
| Synonyms | Handles well | Requires exact terms |
| Speed | Slower (embedding) | Faster |
| Index Size | Larger | Smaller |

# 1.3 Reciprocal Rank Fusion (RRF)

Reciprocal Rank Fusion (RRF) is a technique to combine rankings from multiple retrieval methods. It leverages the strengths of both dense (semantic) and sparse (keyword-based) retrieval systems.

**RRF Formula:**
$$\text{RRF\_score}(d) = \sum_{r \in R} \frac{1}{k + \text{rank}_r(d)}$$

Where:
- $d$ = document
- $R$ = set of ranking methods (Dense + BM25)
- $\text{rank}_r(d)$ = rank of document $d$ in ranking $r$
- $k$ = constant (typically 60) to mitigate the impact of high rankings

**Advantages of RRF:**
- Simple and effective fusion method
- Does not require score normalization across methods
- Robust to outliers in individual rankings
- Captures complementary signals from semantic and keyword-based retrieval

## RRF Implementation

This cell implements the Reciprocal Rank Fusion function that:
1. Retrieves top-K chunks from Dense Vector retrieval
2. Retrieves top-K chunks from BM25 Sparse retrieval
3. Combines rankings using the RRF formula with k=60
4. Returns top-N documents by RRF score for final context

In [27]:
def reciprocal_rank_fusion(query, top_k=10, top_n=5, rrf_k=60):
    """
    Combines Dense Vector and BM25 retrieval results using Reciprocal Rank Fusion.
    
    RRF_score(d) = Σ 1/(k + rank_i(d)) where k=60 (default)
    
    Args:
        query (str): The search query string
        top_k (int): Number of top results to retrieve from each method (default: 10)
        top_n (int): Number of final results to return after fusion (default: 5)
        rrf_k (int): The k constant in RRF formula (default: 60)
    
    Returns:
        list: List of dictionaries containing RRF scores and documents, sorted by RRF score
    """
    # Retrieve top-K from Dense Vector retrieval
    dense_results = retrieve_top_k(query, embedding_model, faiss_index, corpus, k=top_k)
    
    # Retrieve top-K from BM25 Sparse retrieval
    bm25_results = retrieve_top_k_bm25(query, bm25_index, corpus, k=top_k)
    
    # Create a dictionary to store RRF scores for each document
    # Using document index as key (found by matching with corpus)
    rrf_scores = {}
    doc_info = {}  # Store document info and individual scores
    
    # Process Dense Vector results
    for result in dense_results:
        # Find document index in corpus
        doc = result['document']
        doc_idx = corpus.index(doc) if doc in corpus else None
        
        if doc_idx is not None:
            rank = result['rank']
            rrf_contribution = 1.0 / (rrf_k + rank)
            
            if doc_idx not in rrf_scores:
                rrf_scores[doc_idx] = 0.0
                doc_info[doc_idx] = {
                    'document': doc,
                    'dense_rank': None,
                    'dense_score': None,
                    'bm25_rank': None,
                    'bm25_score': None
                }
            
            rrf_scores[doc_idx] += rrf_contribution
            doc_info[doc_idx]['dense_rank'] = rank
            doc_info[doc_idx]['dense_score'] = result['similarity_score']
    
    # Process BM25 results
    for result in bm25_results:
        doc = result['document']
        doc_idx = corpus.index(doc) if doc in corpus else None
        
        if doc_idx is not None:
            rank = result['rank']
            rrf_contribution = 1.0 / (rrf_k + rank)
            
            if doc_idx not in rrf_scores:
                rrf_scores[doc_idx] = 0.0
                doc_info[doc_idx] = {
                    'document': doc,
                    'dense_rank': None,
                    'dense_score': None,
                    'bm25_rank': None,
                    'bm25_score': None
                }
            
            rrf_scores[doc_idx] += rrf_contribution
            doc_info[doc_idx]['bm25_rank'] = rank
            doc_info[doc_idx]['bm25_score'] = result['bm25_score']
    
    # Sort documents by RRF score and select top-N
    sorted_doc_indices = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)[:top_n]
    
    # Build final results
    final_results = []
    for rank, doc_idx in enumerate(sorted_doc_indices, 1):
        final_results.append({
            'rank': rank,
            'rrf_score': rrf_scores[doc_idx],
            'document': doc_info[doc_idx]['document'],
            'dense_rank': doc_info[doc_idx]['dense_rank'],
            'dense_score': doc_info[doc_idx]['dense_score'],
            'bm25_rank': doc_info[doc_idx]['bm25_rank'],
            'bm25_score': doc_info[doc_idx]['bm25_score']
        })
    
    return final_results


def display_rrf_results(query, results, top_k, rrf_k=60):
    """
    Displays the RRF retrieval results in a formatted manner.
    
    Args:
        query (str): The search query
        results (list): List of RRF retrieval results
        top_k (int): Number of documents retrieved from each method
        rrf_k (int): The k constant used in RRF formula
    """
    print(f"\n{'=' * 80}")
    print(f"RECIPROCAL RANK FUSION RESULTS")
    print(f"{'=' * 80}")
    print(f"Query: \"{query}\"")
    print(f"Parameters: top-K={top_k} from each method, RRF k={rrf_k}")
    print(f"{'=' * 80}")
    
    for result in results:
        print(f"\n[RRF Rank {result['rank']}] RRF Score: {result['rrf_score']:.6f}")
        
        # Show contribution from each method
        contributions = []
        if result['dense_rank'] is not None:
            dense_contrib = 1.0 / (rrf_k + result['dense_rank'])
            contributions.append(f"Dense: rank {result['dense_rank']} → {dense_contrib:.6f}")
        else:
            contributions.append("Dense: not in top-K")
            
        if result['bm25_rank'] is not None:
            bm25_contrib = 1.0 / (rrf_k + result['bm25_rank'])
            contributions.append(f"BM25: rank {result['bm25_rank']} → {bm25_contrib:.6f}")
        else:
            contributions.append("BM25: not in top-K")
        
        print(f"  Contributions: {' | '.join(contributions)}")
        
        doc = result['document']
        title = doc['url'].split('/')[-1].replace('_', ' ')
        print(f"  Title: {title}")
        print(f"  URL: {doc['url']}")
        content_preview = doc['content'][:150] + "..." if len(doc['content']) > 150 else doc['content']
        print(f"  Content: {content_preview}")

## Test RRF Hybrid Retrieval

This cell demonstrates the Reciprocal Rank Fusion system that combines dense vector and BM25 sparse retrieval. The RRF score shows how documents are ranked based on contributions from both retrieval methods.

In [28]:
# Test queries for RRF hybrid retrieval
rrf_test_queries = [
    "famous scientist who developed the theory of relativity",
    "natural wonder canyon in Arizona",
    "Renaissance painting masterpiece",
    "artificial intelligence and machine learning technology",
    "major world war military conflict"
]

# RRF parameters
TOP_K = 10  # Retrieve top-10 from each method
TOP_N = 5   # Return top-5 after fusion
RRF_K = 60  # Standard RRF constant

print("RECIPROCAL RANK FUSION (RRF) HYBRID RETRIEVAL")
print("=" * 80)
print(f"Method: Combining Dense Vector + BM25 Sparse Retrieval")
print(f"RRF Formula: RRF_score(d) = Σ 1/(k + rank_i(d))")
print(f"Parameters: k={RRF_K}, top-K={TOP_K} from each method, returning top-{TOP_N}")
print(f"Corpus size: {len(corpus)} documents")

for query in rrf_test_queries:
    results = reciprocal_rank_fusion(query, top_k=TOP_K, top_n=TOP_N, rrf_k=RRF_K)
    display_rrf_results(query, results, TOP_K, RRF_K)

RECIPROCAL RANK FUSION (RRF) HYBRID RETRIEVAL
Method: Combining Dense Vector + BM25 Sparse Retrieval
RRF Formula: RRF_score(d) = Σ 1/(k + rank_i(d))
Parameters: k=60, top-K=10 from each method, returning top-5
Corpus size: 497 documents

RECIPROCAL RANK FUSION RESULTS
Query: "famous scientist who developed the theory of relativity"
Parameters: top-K=10 from each method, RRF k=60

[RRF Rank 1] RRF Score: 0.032787
  Contributions: Dense: rank 1 → 0.016393 | BM25: rank 1 → 0.016393
  Title: Theory of relativity
  URL: https://en.wikipedia.org/wiki/Theory_of_relativity
  Content: The theory of relativity comprises two physics theories by Albert Einstein: special relativity and general relativity, proposed and published in 1905 ...

[RRF Rank 2] RRF Score: 0.032002
  Contributions: Dense: rank 3 → 0.015873 | BM25: rank 2 → 0.016129
  Title: Albert Einstein
  URL: https://en.wikipedia.org/wiki/Albert_Einstein
  Content: Albert Einstein[a] (14 March 1879 – 18 April 1955) was a German-born the

## Summary: Reciprocal Rank Fusion (RRF)

This section implemented a complete Hybrid Retrieval system using Reciprocal Rank Fusion:

1. **Dual Retrieval**: Retrieve top-K chunks from both Dense Vector and BM25 Sparse methods
2. **RRF Scoring**: Combine rankings using RRF formula: `RRF_score(d) = Σ 1/(k + rank_i(d))` where k=60
3. **Final Selection**: Select top-N chunks by RRF score for final context

**Key Benefits of RRF Hybrid Retrieval:**
- Combines semantic understanding (Dense) with exact keyword matching (BM25)
- No need to normalize scores across different retrieval methods
- Documents appearing in both rankings get boosted scores
- More robust retrieval than using either method alone

| Component | Description |
|-----------|-------------|
| Dense Retrieval | Captures semantic similarity and synonyms |
| BM25 Retrieval | Handles exact keyword matches efficiently |
| RRF Fusion | Rank-based combination, k=60 dampens high rankings |
| Output | Top-N documents with provenance from both methods |

# 1.4 Response Generation

This section implements Response Generation using **TinyLlama-1.1B-Chat** with 4-bit GGUF quantization. The approach involves:
1. **Model Loading**: Load TinyLlama-1.1B-Chat GGUF model with 4-bit quantization using ctransformers
2. **Context Construction**: Concatenate top-N retrieved chunks with the query
3. **Answer Generation**: Generate responses using ChatML-formatted prompts

**TinyLlama-1.1B-Chat Characteristics:**
- Compact 1.1B parameter model optimized for chat
- Fast inference on CPU (~10-20 seconds per query)
- 4-bit GGUF quantization (Q4_K_M) reduces model size to ~700MB
- Maximum context length: 2048 tokens
- Uses ctransformers for efficient CPU inference on Windows

In [29]:
# Install prerequisites for GGUF model loading (CPU version)
# ctransformers: Efficient library for running GGUF models
# huggingface_hub: For downloading models from HuggingFace

import subprocess
import sys

def install_package(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

print("Installing prerequisites for Llama-2-7B-Chat GGUF (CPU version)...")

# Install ctransformers WITHOUT CUDA for CPU-only usage (smaller package, faster install)
install_package("ctransformers>=0.2.27")
install_package("huggingface_hub>=0.19.0")

print("Prerequisites installed successfully!")

Installing prerequisites for Llama-2-7B-Chat GGUF (CPU version)...
Prerequisites installed successfully!


## Load TinyLlama-1.1B-Chat with 4-bit GGUF Quantization (CPU)

This cell loads the TinyLlama-1.1B-Chat model with 4-bit GGUF quantization optimized for CPU:
- Uses `ctransformers` for efficient CPU inference on Windows
- Downloads the GGUF model from HuggingFace Hub if not present
- **4-bit quantization (Q4_K_M)** - compact ~700MB download, 6x smaller than Llama-2-7B
- **TinyLlama-Chat** is optimized for fast conversational AI on CPU
- Set `gpu_layers=0` for CPU-only execution
- Expected inference speed: ~10-20 seconds per query

In [30]:
from ctransformers import AutoModelForCausalLM
import os

# =============================================================================
# USING TINYLLAMA-1.1B - 6x smaller than Llama-2-7B, much faster on CPU!
# =============================================================================
# TinyLlama is a 1.1B parameter model that runs FAST on CPU
# Expected: ~10-20 seconds per query instead of 2-3 minutes

LLM_MODEL_NAME = "TinyLlama-1.1B-Chat-GGUF"
LLM_MODEL_REPO = "TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF"
LLM_MODEL_FILE = "tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf"  # ~700MB download
MAX_CONTEXT_LENGTH = 2048

# CPU Optimization: Use all available threads
CPU_THREADS = os.cpu_count() or 4

print(f"Loading {LLM_MODEL_NAME} (FAST CPU model)...")
print(f"Model repository: {LLM_MODEL_REPO}")
print(f"Model file: {LLM_MODEL_FILE}")
print(f"Model size: ~700MB (vs 4GB for Llama-2-7B)")
print(f"CPU Threads: {CPU_THREADS}")
print("Downloading model if not cached...")

# Load TinyLlama with CPU optimizations
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_REPO,
    model_file=LLM_MODEL_FILE,
    model_type="llama",
    context_length=MAX_CONTEXT_LENGTH,
    gpu_layers=0,
    threads=CPU_THREADS,
    batch_size=512
)

print(f"\n✓ Model loaded successfully!")
print(f"Model: {LLM_MODEL_NAME} (1.1B parameters)")
print(f"Expected speed: ~10-20 seconds per query on CPU")

Loading TinyLlama-1.1B-Chat-GGUF (FAST CPU model)...
Model repository: TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF
Model file: tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf
Model size: ~700MB (vs 4GB for Llama-2-7B)
CPU Threads: 12


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]c:\Users\koush\OneDrive\Documents\BITS Pilani\Conversational AI\Assignment 2\.venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\koush\.cache\huggingface\hub\models--TheBloke--TinyLlama-1.1B-Chat-v1.0-GGUF. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(m


✓ Model loaded successfully!
Model: TinyLlama-1.1B-Chat-GGUF (1.1B parameters)
Expected speed: ~10-20 seconds per query on CPU


## Context Construction Function

This cell implements functions to:
1. Build a context string from retrieved chunks
2. Truncate context to fit within model's token limit
3. Format the prompt using **ChatML format** with `<|system|>`, `<|user|>`, and `<|assistant|>` tags
4. Apply strict system instructions for **context-only answering**

In [31]:
def build_context_from_chunks(retrieved_results, max_chars=800):
    """
    Builds a context string from retrieved document chunks.
    Optimized: reduced max_chars for faster inference.
    """
    context_parts = []
    total_chars = 0
    
    for result in retrieved_results:
        doc = result['document']
        content = doc['content']
        
        if total_chars + len(content) > max_chars:
            remaining_chars = max_chars - total_chars
            if remaining_chars > 100:
                context_parts.append(content[:remaining_chars] + "...")
            break
        
        context_parts.append(content)
        total_chars += len(content)
    
    return "\n\n".join(context_parts)


def create_prompt(query, context):
    """
    Creates a ChatML formatted prompt for TinyLlama-Chat.
    TinyLlama uses ChatML format, not Llama-2 format.
    """
    # TinyLlama-Chat uses ChatML format
    prompt = f"""<|system|>
You are a helpful assistant. Answer questions using ONLY the provided context. If the context doesn't contain the answer, say "I cannot answer based on the provided context." Be concise.</s>
<|user|>
Context: {context}

Question: {query}</s>
<|assistant|>
"""
    return prompt


print("Context construction functions defined successfully!")
print("Using ChatML prompt format for TinyLlama-Chat.")

Context construction functions defined successfully!
Using ChatML prompt format for TinyLlama-Chat.


## Response Generation Function

This cell implements the main response generation function that:
1. Retrieves top-N chunks using RRF hybrid retrieval
2. Constructs context from retrieved chunks
3. Generates an answer using TinyLlama-1.1B-Chat with ChatML-formatted prompts

In [32]:
def generate_response(query, top_k=5, top_n=2, max_tokens=80, temperature=0.1):
    """
    Generates a response using TinyLlama-1.1B - optimized for CPU speed.
    Expected: ~10-20 seconds per query.
    
    Returns detailed results including dense, sparse, and RRF scores for each chunk.
    """
    import time
    start_time = time.time()
    
    # Step 1: Get individual retrieval results for scoring details
    dense_results = retrieve_top_k(query, embedding_model, faiss_index, corpus, k=top_k)
    bm25_results = retrieve_top_k_bm25(query, bm25_index, corpus, k=top_k)
    
    # Step 2: Retrieve chunks using RRF (hybrid fusion)
    retrieved_results = reciprocal_rank_fusion(query, top_k=top_k, top_n=top_n, rrf_k=60)
    retrieval_time = time.time() - start_time
    
    # Step 3: Build compact context
    context = build_context_from_chunks(retrieved_results, max_chars=600)
    
    # Step 4: Create ChatML formatted prompt for TinyLlama
    prompt = create_prompt(query, context)
    
    # Step 5: Generate response
    gen_start = time.time()
    answer = llm_model(
        prompt,
        max_new_tokens=max_tokens,
        temperature=temperature,
        top_p=0.9,
        top_k=40,
        repetition_penalty=1.1,
        stop=["</s>", "<|user|>", "<|system|>", "\n\n\n"]
    )
    gen_time = time.time() - gen_start
    
    # Clean up answer
    answer = answer.strip()
    answer = answer.replace("</s>", "").replace("<|", "").strip()
    
    # Truncate at complete sentence if needed
    if answer and not answer.endswith(('.', '!', '?')):
        last_period = max(answer.rfind('.'), answer.rfind('!'), answer.rfind('?'))
        if last_period > len(answer) * 0.3:
            answer = answer[:last_period + 1]
    
    total_time = time.time() - start_time
    
    return {
        'query': query,
        'retrieved_chunks': retrieved_results,
        'dense_results': dense_results,
        'bm25_results': bm25_results,
        'context': context,
        'answer': answer,
        'timing': {
            'retrieval_sec': round(retrieval_time, 2),
            'generation_sec': round(gen_time, 2),
            'total_sec': round(total_time, 2)
        }
    }


def display_generation_results(result):
    """
    Displays comprehensive response generation results including:
    - User query input
    - Generated answer
    - Top retrieved chunks with sources
    - Dense/Sparse/RRF scores
    - Response time breakdown
    """
    print(f"\n{'=' * 80}")
    print(f"RAG RESPONSE GENERATION RESULTS")
    print(f"{'=' * 80}")
    
    # 1. User Query Input
    print(f"\n📝 USER QUERY:")
    print(f"   {result['query']}")
    
    # 2. Generated Answer
    print(f"\n🤖 GENERATED ANSWER:")
    print(f"   {result['answer']}")
    
    # 3. Top Retrieved Chunks with Sources and Scores
    print(f"\n📚 TOP RETRIEVED CHUNKS ({len(result['retrieved_chunks'])} chunks):")
    print(f"   {'─' * 70}")
    
    for i, chunk in enumerate(result['retrieved_chunks'], 1):
        doc = chunk['document']
        title = doc['url'].split('/')[-1].replace('_', ' ')
        source_url = doc['url']
        domain = doc['domain']
        
        # Get scores
        rrf_score = chunk.get('rrf_score', 0)
        dense_rank = chunk.get('dense_rank', 'N/A')
        dense_score = chunk.get('dense_score', 'N/A')
        bm25_rank = chunk.get('bm25_rank', 'N/A')
        bm25_score = chunk.get('bm25_score', 'N/A')
        
        print(f"\n   [{i}] {title}")
        print(f"       📖 Source: {source_url}")
        print(f"       🏷️  Domain: {domain}")
        print(f"       📊 Scores:")
        
        # Dense score
        if dense_score != 'N/A' and dense_score is not None:
            print(f"          • Dense (Semantic):  Rank {dense_rank}, Score: {dense_score:.4f}")
        else:
            print(f"          • Dense (Semantic):  Not in top-K")
        
        # BM25 score
        if bm25_score != 'N/A' and bm25_score is not None:
            print(f"          • Sparse (BM25):     Rank {bm25_rank}, Score: {bm25_score:.4f}")
        else:
            print(f"          • Sparse (BM25):     Not in top-K")
        
        # RRF score
        print(f"          • RRF (Hybrid):      Score: {rrf_score:.6f}")
        
        # Content preview
        content_preview = doc['content'][:150] + "..." if len(doc['content']) > 150 else doc['content']
        print(f"       📄 Content Preview: {content_preview}")
    
    # 4. Response Time Breakdown
    t = result['timing']
    print(f"\n⏱️  RESPONSE TIME BREAKDOWN:")
    print(f"   {'─' * 40}")
    print(f"   • Retrieval Time:   {t['retrieval_sec']:>6.2f} seconds")
    print(f"   • Generation Time:  {t['generation_sec']:>6.2f} seconds")
    print(f"   • Total Time:       {t['total_sec']:>6.2f} seconds")
    
    print(f"\n{'=' * 80}")


print("✓ Response generation functions ready!")
print("Using TinyLlama-1.1B - expect ~10-20 sec/query on CPU")
print("\nOutput includes: Query, Answer, Retrieved Chunks with Sources, Dense/Sparse/RRF Scores, Response Time")

✓ Response generation functions ready!
Using TinyLlama-1.1B - expect ~10-20 sec/query on CPU

Output includes: Query, Answer, Retrieved Chunks with Sources, Dense/Sparse/RRF Scores, Response Time


## Test Response Generation

This cell demonstrates the complete RAG (Retrieval-Augmented Generation) pipeline:
1. Query is processed through RRF hybrid retrieval
2. Top-N chunks are concatenated as context
3. TinyLlama-1.1B-Chat generates an answer grounded in the context

In [33]:
# Test with just ONE query first to verify speed
rag_test_queries = [
    "Define photosynthesis"
]

print("RAG DEMONSTRATION - TinyLlama-1.1B (Fast CPU Model)")
print("=" * 70)
print(f"Model: {LLM_MODEL_NAME}")
print(f"Size: 1.1B parameters (~700MB)")
print(f"Expected: 10-20 seconds per query")
print("=" * 70)

# Test single query
for query in rag_test_queries:
    print(f"\n🔄 Processing: '{query}'")
    result = generate_response(query, top_k=5, top_n=2, max_tokens=80)
    display_generation_results(result)

RAG DEMONSTRATION - TinyLlama-1.1B (Fast CPU Model)
Model: TinyLlama-1.1B-Chat-GGUF
Size: 1.1B parameters (~700MB)
Expected: 10-20 seconds per query

🔄 Processing: 'Define photosynthesis'

RAG RESPONSE GENERATION RESULTS

📝 USER QUERY:
   Define photosynthesis

🤖 GENERATED ANSWER:
   Photosynthesis is a biochemical process by which plants, algae, and cyanobacteria convert light energy into chemical energy in the form of organic compounds. This process involves the conversion of water (H2O) into oxygen (O2) and carbon dioxide (CO2).

📚 TOP RETRIEVED CHUNKS (2 chunks):
   ──────────────────────────────────────────────────────────────────────

   [1] Photosynthesis
       📖 Source: https://en.wikipedia.org/wiki/Photosynthesis
       🏷️  Domain: science
       📊 Scores:
          • Dense (Semantic):  Rank 1, Score: 0.6873
          • Sparse (BM25):     Rank 1, Score: 11.5081
          • RRF (Hybrid):      Score: 0.032787
       📄 Content Preview: Photosynthesis (/ˌfoʊtəˈsɪnθəsɪs/ FOH-tə-SI

## Summary: Response Generation

This section implemented a complete Response Generation system using TinyLlama-1.1B-Chat:

1. **Model Loading**: TinyLlama-1.1B-Chat loaded with 4-bit GGUF quantization (~700MB)
2. **Context Construction**: Top-N retrieved chunks concatenated within context limits
3. **Answer Generation**: TinyLlama generates contextual answers using ChatML-formatted prompts

**Key Components:**
| Component | Description |
|-----------|-------------|
| Model | TinyLlama-1.1B-Chat (1.1B parameters) |
| Quantization | 4-bit GGUF (Q4_K_M) |
| Max Context | 2048 tokens |
| Retrieval | RRF hybrid (Dense + BM25) |
| Generation | Temperature=0.1, top-p=0.9 |

**RAG Pipeline Flow:**
1. Query → RRF Retrieval → Top-N Chunks
2. Chunks → Context Construction → Truncated Context  
3. Context + Query → ChatML Prompt → Answer

**Advantages of this approach:**
- Compact 1.1B parameter model optimized for fast CPU inference
- Grounded answers based on retrieved Wikipedia content
- 4-bit quantization enables running on consumer hardware (~700MB)
- Combines semantic and keyword-based retrieval for better context
- Low temperature (0.1) ensures focused, factual responses
- Fast inference: ~10-20 seconds per query on CPU